In [ ]:
# JupyterLite 环境准备：安装本题依赖，并按需下载本题素材
from pathlib import Path
import hashlib
import piplite
from pyodide.http import pyfetch

PACKAGES = ["numpy", "scipy", "Pillow"]
ASSETS = [{"name": "img_test.jpg", "url": "/labs/3.2.1/img_test.jpg", "size": 150351, "sha256": "d39697e9ee9f37371e13f50c33e6677275fe28e11839173d37ac98cf432d83d1"}, {"name": "labels.txt", "url": "/labs/3.2.1/labels.txt", "size": 10472, "sha256": "1f386e0d1cb6e28b9c2dac651c3dea6801e98ad1b41a14ce6bb1a093d72069f5"}]

if PACKAGES:
    await piplite.install(PACKAGES)

for asset in ASSETS:
    target = Path(asset["name"])
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() == asset["sha256"]:
        continue
    response = await pyfetch(asset["url"])
    payload = await response.bytes()
    digest = hashlib.sha256(payload).hexdigest()
    if digest != asset["sha256"]:
        raise RuntimeError(f"素材校验失败：{asset['name']}")
    target.write_bytes(payload)

print("✓ 3.2.1 环境已就绪", f"依赖 {len(PACKAGES)} 项，素材 {len(ASSETS)} 个")


In [ ]:
# JupyterLite 浏览器 ONNX Runtime：从现有分片清单加载、校验、缓存并执行模型
from pyodide.code import run_js

_browser_ort_run = run_js(r"""
async (taskCode) => {
  const origin = self.location.origin;
  const digest = async buffer => {
    const hash = await crypto.subtle.digest("SHA-256", buffer);
    return [...new Uint8Array(hash)].map(value => value.toString(16).padStart(2, "0")).join("");
  };
  const response = await fetch(`${origin}/labs/manifest.json`);
  if (!response.ok) throw new Error("模型清单加载失败");
  const manifest = await response.json();
  const model = manifest.tasks?.[taskCode]?.model;
  if (!model) throw new Error(`任务 ${taskCode} 没有浏览器模型`);

  const cache = await caches.open("ai-trainer-onnx-v2");
  const cacheKey = new Request(`${origin}/labs/model-cache/${model.modelId}.onnx`);
  let modelBytes;
  const cached = await cache.match(cacheKey);
  if (cached) {
    const cachedBuffer = await cached.arrayBuffer();
    if (await digest(cachedBuffer) === model.sha256) modelBytes = new Uint8Array(cachedBuffer);
  }
  if (!modelBytes) {
    const chunks = [];
    let total = 0;
    for (const part of model.chunks) {
      const partResponse = await fetch(`${origin}${part.url}`);
      if (!partResponse.ok) throw new Error(`模型分片加载失败：${part.url}`);
      const partBuffer = await partResponse.arrayBuffer();
      if (await digest(partBuffer) !== part.sha256) throw new Error(`模型分片校验失败：${part.url}`);
      const bytes = new Uint8Array(partBuffer);
      chunks.push(bytes);
      total += bytes.byteLength;
    }
    modelBytes = new Uint8Array(total);
    let offset = 0;
    for (const chunk of chunks) { modelBytes.set(chunk, offset); offset += chunk.byteLength; }
    if (await digest(modelBytes.buffer) !== model.sha256) throw new Error("模型完整性校验失败");
    await cache.put(cacheKey, new Response(modelBytes, { headers: { "content-type": "application/octet-stream" } }));
  }

  const runtimeBase = `${origin}/labs/runtime/onnxruntime-web/1.27.0/`;
  const ort = await import(`${runtimeBase}ort.wasm.bundle.min.js`);
  ort.env.wasm.numThreads = 1;
  ort.env.wasm.wasmPaths = runtimeBase;
  const session = await ort.InferenceSession.create(modelBytes, { executionProviders: ["wasm"] });
  const feeds = {};
  const inputs = [];
  for (const [index, name] of session.inputNames.entries()) {
    const meta = session.inputMetadata[index];
    if (!meta?.isTensor) continue;
    const shape = (meta.shape?.length ? meta.shape : [1]).map(value => Number.isInteger(value) && value > 0 ? value : 1);
    const size = shape.reduce((left, right) => left * right, 1);
    const sourceType = String(meta.type ?? "float32");
    const type = sourceType.includes("uint8") ? "uint8" : sourceType.includes("int32") ? "int32" : "float32";
    const data = type === "uint8" ? new Uint8Array(size) : type === "int32" ? new Int32Array(size) : new Float32Array(size);
    feeds[name] = new ort.Tensor(type, data, shape);
    inputs.push({ name, type, shape });
  }
  const outputMap = await session.run(feeds);
  const outputs = Object.entries(outputMap).map(([name, tensor]) => ({ name, type: tensor.type, shape: [...tensor.dims] }));
  return { model: model.name, runtime: model.browserRuntime, variant: model.sourceVariant, inputs, outputs };
}
""")

async def browser_onnx_check(task_code="3.2.1"):
    """执行一次真实的浏览器 WASM 推理，并返回张量结构。"""
    result = await _browser_ort_run(task_code)
    return result.to_py() if hasattr(result, "to_py") else result

onnx_result = await browser_onnx_check()
onnx_result


In [ ]:
import onnxruntime as ort
import numpy as np
import scipy.special
from PIL import Image


# 预处理图像
def preprocess_image(image, resize_size=256, crop_size=224, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
  image = image.resize((resize_size, resize_size), Image.BILINEAR)
  w, h = image.size
  left = (w - crop_size) / 2
  top = (h - crop_size) / 2
  image = image.crop((left, top, left + crop_size, top + crop_size))
  image = np.array(image).astype(np.float32)
  image = image / 255.0
  image = (image - mean) / std
  image = np.transpose(image, (2, 0, 1))
  image = image.reshape((1,) + image.shape)
  return image


# 模型加载 2分
session = _________________


# 加载类别标签
labels_path = 'labels.txt'
with open(labels_path) as f:
  labels = [line.strip() for line in f.readlines()]


# 获取模型输入和输出的名称
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name


# 加载图片 2分
image = _________________('RGB')


# 预处理图片 2分
processed_image = _________________


# 确保输入数据是 float32 类型
processed_image = processed_image.astype(np.float32)


# 进行图片识别 2分
output = _________________([output_name], {input_name: processed_image})[0]


# 应用 softmax 函数获取概率 2分
probabilities = _________________(output, axis=-1)


# 获取最高的5个概率和对应的类别索引 3分
top5_idx = _________________[-5:][::-1]
top5_prob = _________________


# 打印结果
print("Top 5 predicted classes:")
for i in range(5):
  print(f"{i+1}: {labels[top5_idx[i]]} - Probability: {top5_prob[i]}")
